# Module 00: Power Automate via the Microsoft Graph API

## Learning Objectives

By completing this notebook, you will:

1. Authenticate against the Microsoft Graph API using client credentials (OAuth 2.0)
2. List Power Automate flows in a tenant using the Graph API
3. Inspect flow metadata — trigger type, connection references, status, and owner
4. Retrieve flow run history and parse execution outcomes programmatically
5. Build a reusable API client class for Power Automate operations

## Why This Matters

The Power Automate portal is the visual interface, but the **Microsoft Graph API** and the **Power Automate Management API** are what enable:

- Automated flow auditing across a large tenant
- Bulk enable/disable operations
- Integration with CI/CD pipelines
- Monitoring dashboards that aggregate run history
- Governance scripts (find flows using deprecated connectors, identify orphaned flows, etc.)

Understanding the API layer makes you a significantly more effective Power Automate practitioner.

## Prerequisites

- Python 3.9+
- A Microsoft Azure AD tenant with permission to register an app
- `requests` library installed

## Estimated Time: 15 minutes

## Setup

Install the required library if not already installed:

```bash
pip install requests
```

In [ ]:
learning_objectives([
    "and the **Power Automate Management API** are what enable:",
    "Automated flow auditing across a large tenant",
    "Bulk enable/disable operations",
    "Integration with CI/CD pipelines",
    "Monitoring dashboards that aggregate run history",
    "Governance scripts (find flows using deprecated connectors, identify orphaned flows, etc.)",
    "Python 3.9+",
    "A Microsoft Azure AD tenant with permission to register an app",
    "`requests` library installed"
])

---

## Section 1: Azure AD App Registration

Before the code runs, you need an Azure AD application registration with the correct permissions. This is a one-time setup.

### Steps to Register an App

1. Go to `https://portal.azure.com` and sign in with your organizational account
2. Navigate to **Azure Active Directory** → **App registrations** → **New registration**
3. Fill in:
   - **Name:** `PowerAutomate-API-Explorer` (or any descriptive name)
   - **Supported account types:** Accounts in this organizational directory only (single tenant)
   - **Redirect URI:** Leave blank for now (we use client credentials flow, not user auth)
4. Click **Register**
5. Copy the **Application (client) ID** and **Directory (tenant) ID** from the Overview page
6. Go to **Certificates & secrets** → **New client secret**
   - Description: `notebook-secret`
   - Expiry: 6 months
   - Click **Add** and **immediately copy the Value** (it is only shown once)

### Required API Permissions

7. Go to **API permissions** → **Add a permission** → **Microsoft Graph** → **Application permissions**
8. Add these permissions:
   - `Flow.Read.All` — read all flows in the tenant
   - `User.Read.All` — resolve flow owner display names
9. Click **Grant admin consent for [your org]** (requires a Global Admin or Privileged Role Admin)

> **Note on Permissions:** `Flow.Read.All` is a Graph API permission that gives read access to Power Automate flows. If your organization has not enabled this permission scope, the flows endpoint will return an empty list even with a valid token. Work with your Azure AD administrator if you encounter this.

### Alternative: Power Automate Management API

For environments where Graph API flow permissions are restricted, the **Power Automate Management API** (`management.azure.com`) provides an alternative path. Section 4 of this notebook covers that endpoint.

In [ ]:
import sys; sys.path.insert(0, '../../../../..')
from resources.notebook_style import apply_course_theme, learning_objectives, section_divider, callout, key_takeaways
from resources.graphics.plot_theme import apply_plot_theme

In [ ]:
apply_course_theme()
apply_plot_theme()

In [ ]:
# Configuration — replace these values with your app registration details
# Store these in environment variables in production; never commit secrets to source control

import os

# Read from environment variables (preferred) or set directly for local notebook use
TENANT_ID = os.environ.get("AZURE_TENANT_ID", "your-tenant-id-here")
CLIENT_ID = os.environ.get("AZURE_CLIENT_ID", "your-client-id-here")
CLIENT_SECRET = os.environ.get("AZURE_CLIENT_SECRET", "your-client-secret-here")

# The environment ID for the Power Automate environment you want to inspect
# Find this in the Power Automate portal: Settings > Session details, or
# in the URL when navigating to an environment: make.powerautomate.com/environments/{ENVIRONMENT_ID}/
ENVIRONMENT_ID = os.environ.get("PA_ENVIRONMENT_ID", "your-environment-id-here")

# Validate that configuration is set (fail fast rather than silent errors later)
missing = [
    name for name, val in [
        ("TENANT_ID", TENANT_ID),
        ("CLIENT_ID", CLIENT_ID),
        ("CLIENT_SECRET", CLIENT_SECRET),
        ("ENVIRONMENT_ID", ENVIRONMENT_ID),
    ]
    if val.endswith("-here")
]

if missing:
    print(f"[SETUP NEEDED] Replace placeholder values for: {', '.join(missing)}")
    print("See Section 1 for Azure AD app registration instructions.")
else:
    print("[OK] All configuration values are set.")
    print(f"Tenant: {TENANT_ID[:8]}...")
    print(f"Client: {CLIENT_ID[:8]}...")

---

## Section 2: OAuth 2.0 Authentication

Power Automate APIs use **OAuth 2.0 with the client credentials grant** for server-to-server (daemon) access. This means:

- No user login required
- The app authenticates as itself, using its own credentials
- Access is governed by the API permissions granted in Azure AD (not individual user permissions)

The token endpoint follows this pattern:

```
POST https://login.microsoftonline.com/{tenant_id}/oauth2/v2.0/token

Body (form-encoded):
  client_id={client_id}
  client_secret={client_secret}
  scope=https://graph.microsoft.com/.default
  grant_type=client_credentials
```

The response includes an `access_token` (a JWT) that expires in 3600 seconds (1 hour). All subsequent API calls include this token in the `Authorization: Bearer {token}` header.

In [ ]:
import requests
import json
from datetime import datetime, timezone
from typing import Optional


class GraphAPIClient:
    """
    A minimal client for the Microsoft Graph API focused on Power Automate resources.

    Handles:
    - OAuth 2.0 client credentials authentication
    - Automatic token refresh when expired
    - Pagination of list responses (OData @odata.nextLink)
    - Structured error reporting
    """

    GRAPH_BASE = "https://graph.microsoft.com/v1.0"
    TOKEN_URL_TEMPLATE = "https://login.microsoftonline.com/{tenant_id}/oauth2/v2.0/token"
    GRAPH_SCOPE = "https://graph.microsoft.com/.default"

    def __init__(self, tenant_id: str, client_id: str, client_secret: str):
        self.tenant_id = tenant_id
        self.client_id = client_id
        self.client_secret = client_secret
        self._access_token: Optional[str] = None
        self._token_expiry: Optional[float] = None

    def _is_token_valid(self) -> bool:
        """Check if the cached token exists and has not expired (with 60s buffer)."""
        if self._access_token is None or self._token_expiry is None:
            return False
        now = datetime.now(timezone.utc).timestamp()
        # 60-second buffer prevents edge-case failures when token expires mid-request
        return now < (self._token_expiry - 60)

    def authenticate(self) -> None:
        """
        Acquire an OAuth 2.0 access token using the client credentials grant.

        Stores the token and its expiry time for reuse. Call this explicitly
        or let get() call it automatically via ensure_authenticated().
        """
        token_url = self.TOKEN_URL_TEMPLATE.format(tenant_id=self.tenant_id)
        payload = {
            "client_id": self.client_id,
            "client_secret": self.client_secret,
            "scope": self.GRAPH_SCOPE,
            "grant_type": "client_credentials",
        }

        response = requests.post(token_url, data=payload, timeout=30)

        if response.status_code != 200:
            error = response.json()
            raise RuntimeError(
                f"Authentication failed: {error.get('error')} — {error.get('error_description')}"
            )

        token_data = response.json()
        self._access_token = token_data["access_token"]
        # expires_in is seconds from now; convert to absolute timestamp
        self._token_expiry = (
            datetime.now(timezone.utc).timestamp() + token_data["expires_in"]
        )

        print(f"[AUTH] Token acquired. Expires in {token_data['expires_in']} seconds.")

    def _ensure_authenticated(self) -> None:
        """Acquire or refresh the token if it is missing or expired."""
        if not self._is_token_valid():
            self.authenticate()

    def get(self, path: str, params: Optional[dict] = None) -> dict:
        """
        Make a GET request to the Graph API.

        Args:
            path: API path relative to the Graph base URL, or a full URL
                  (used for @odata.nextLink pagination).
            params: Optional query parameters dict.

        Returns:
            Parsed JSON response body as a dict.

        Raises:
            RuntimeError: On non-2xx HTTP status with the error details.
        """
        self._ensure_authenticated()

        # Support both relative paths and full URLs (for nextLink pagination)
        url = path if path.startswith("https://") else f"{self.GRAPH_BASE}{path}"

        headers = {
            "Authorization": f"Bearer {self._access_token}",
            "Accept": "application/json",
            "Content-Type": "application/json",
        }

        response = requests.get(url, headers=headers, params=params, timeout=30)

        if response.status_code == 401:
            # Token rejected — force refresh and retry once
            self._access_token = None
            self._ensure_authenticated()
            headers["Authorization"] = f"Bearer {self._access_token}"
            response = requests.get(url, headers=headers, params=params, timeout=30)

        if not response.ok:
            raise RuntimeError(
                f"API error {response.status_code} on GET {url}: {response.text[:500]}"
            )

        return response.json()

    def get_all_pages(self, path: str, params: Optional[dict] = None) -> list:
        """
        Retrieve all pages from a paginated OData collection endpoint.

        Graph API lists return up to 100 items per page by default.
        Large tenants with thousands of flows require pagination.

        Returns:
            Flat list of all items across all pages.
        """
        all_items = []
        next_url = path
        page_count = 0

        while next_url:
            page_count += 1
            data = self.get(next_url, params=params if page_count == 1 else None)
            all_items.extend(data.get("value", []))
            next_url = data.get("@odata.nextLink")  # None on the last page

        print(f"[PAGED] Retrieved {len(all_items)} items across {page_count} page(s).")
        return all_items


print("GraphAPIClient class defined.")

In [ ]:
# Instantiate the client — authentication happens lazily on first API call
# (or call client.authenticate() explicitly to test credentials now)

client = GraphAPIClient(
    tenant_id=TENANT_ID,
    client_id=CLIENT_ID,
    client_secret=CLIENT_SECRET,
)

# Test authentication immediately — this confirms credentials before any API calls
# Expected output: "[AUTH] Token acquired. Expires in 3600 seconds."
# Common error: "Authentication failed: invalid_client" — check CLIENT_SECRET value
# Common error: "Authentication failed: unauthorized_client" — app not granted consent
try:
    client.authenticate()
except RuntimeError as e:
    print(f"[ERROR] {e}")
    print("Check your TENANT_ID, CLIENT_ID, and CLIENT_SECRET values.")

---

## Section 3: Listing Flows via the Graph API

The Microsoft Graph API exposes Power Automate flows under the `/v1.0/flows` endpoint (preview).
For production use, the **Power Automate Management API** (`management.azure.com`) is the more
stable and feature-complete option — Section 4 covers that.

The Graph flows endpoint returns flow objects with this structure:

```json
{
  "id": "flow-guid",
  "displayName": "Send Teams message when SharePoint item created",
  "state": "started",
  "createdDateTime": "2024-11-15T10:23:00Z",
  "lastModifiedDateTime": "2025-01-08T14:45:00Z",
  "description": "",
  "connectionReferences": { ... },
  "definition": { ... }
}
```

The `definition` field contains the full flow JSON — triggers, actions, conditions — which we
will parse to extract the trigger type in the next section.

In [ ]:
def list_flows_graph(client: GraphAPIClient, environment_id: str) -> list[dict]:
    """
    List all Power Automate flows in a given environment using the Graph API.

    Endpoint: GET /v1.0/solutions/flows (environment-scoped)

    Note: The Graph API for flows uses an environment-scoped path pattern.
    The environment ID is the same GUID found in the Power Automate portal URL.

    Args:
        client: Authenticated GraphAPIClient instance.
        environment_id: The Power Automate environment ID (GUID).

    Returns:
        List of flow dicts, each containing id, displayName, state, and metadata.
    """
    # Graph API path for flows scoped to a specific environment
    # Full docs: https://learn.microsoft.com/graph/api/flow-list
    path = f"/solutions/flows"

    params = {
        "$filter": f"environment/id eq '{environment_id}'",
        "$select": "id,displayName,state,createdDateTime,lastModifiedDateTime,description",
        "$top": 100,  # Maximum page size
    }

    flows = client.get_all_pages(path, params=params)
    return flows


# Retrieve flows — expected output: list of dicts with flow metadata
# If environment_id is wrong: returns empty list (no error)
# If permissions are missing: RuntimeError with 403 status
try:
    flows = list_flows_graph(client, ENVIRONMENT_ID)
    print(f"Found {len(flows)} flow(s) in environment {ENVIRONMENT_ID[:8]}...")
except RuntimeError as e:
    print(f"[ERROR] {e}")
    flows = []  # Allows downstream cells to handle empty gracefully

In [ ]:
# Inspect the raw structure of the first flow returned
# This teaches what fields are available before we build higher-level functions

if flows:
    first_flow = flows[0]
    print("=== First Flow — Raw Fields ===")
    for key, value in first_flow.items():
        # Truncate long values for readability
        display_value = str(value)
        if len(display_value) > 120:
            display_value = display_value[:120] + "..."
        print(f"  {key:<30} {display_value}")
else:
    print("No flows returned. Check environment ID and API permissions.")
    print("Expected output when working correctly:")
    print("  id                             abc12345-...")
    print("  displayName                    My example flow")
    print("  state                          started")
    print("  createdDateTime                2024-11-15T10:23:00Z")
    print("  lastModifiedDateTime           2025-01-08T14:45:00Z")

---

## Section 4: Power Automate Management API

The **Power Automate Management API** (`api.flow.microsoft.com`) is the primary REST API for
Power Automate operations. It is more complete than the Graph API flows endpoint and is the
backend the portal itself uses.

Key differences from Graph API:

| Aspect | Graph API | Management API |
|---|---|---|
| Base URL | `graph.microsoft.com/v1.0` | `api.flow.microsoft.com` |
| Auth scope | `https://graph.microsoft.com/.default` | `https://service.flow.microsoft.com/.default` |
| Flow detail | Partial | Full (includes definition JSON) |
| Run history | Not available | Available |
| Stability | Preview for flows | GA |

For the Management API, we need a separate token with the `service.flow.microsoft.com` scope.

In [ ]:
class PowerAutomateManagementClient:
    """
    Client for the Power Automate Management API.

    Base URL: https://api.flow.microsoft.com
    Auth scope: https://service.flow.microsoft.com/.default

    Full API reference:
    https://learn.microsoft.com/en-us/power-automate/web-api
    """

    BASE_URL = "https://api.flow.microsoft.com"
    TOKEN_URL_TEMPLATE = "https://login.microsoftonline.com/{tenant_id}/oauth2/v2.0/token"
    # This scope grants access to the Power Automate service, not Graph
    PA_SCOPE = "https://service.flow.microsoft.com/.default"

    def __init__(self, tenant_id: str, client_id: str, client_secret: str, environment_id: str):
        self.tenant_id = tenant_id
        self.client_id = client_id
        self.client_secret = client_secret
        self.environment_id = environment_id
        self._access_token: Optional[str] = None
        self._token_expiry: Optional[float] = None

    def _is_token_valid(self) -> bool:
        if self._access_token is None or self._token_expiry is None:
            return False
        return datetime.now(timezone.utc).timestamp() < (self._token_expiry - 60)

    def authenticate(self) -> None:
        """Acquire an access token for the Power Automate Management API scope."""
        token_url = self.TOKEN_URL_TEMPLATE.format(tenant_id=self.tenant_id)
        payload = {
            "client_id": self.client_id,
            "client_secret": self.client_secret,
            "scope": self.PA_SCOPE,
            "grant_type": "client_credentials",
        }
        response = requests.post(token_url, data=payload, timeout=30)
        if not response.ok:
            error = response.json()
            raise RuntimeError(
                f"PA Management API auth failed: {error.get('error')} — {error.get('error_description')}"
            )
        token_data = response.json()
        self._access_token = token_data["access_token"]
        self._token_expiry = datetime.now(timezone.utc).timestamp() + token_data["expires_in"]
        print(f"[AUTH] PA Management API token acquired. Expires in {token_data['expires_in']}s.")

    def _ensure_authenticated(self) -> None:
        if not self._is_token_valid():
            self.authenticate()

    def _get(self, path: str, params: Optional[dict] = None) -> dict:
        """Execute a GET request against the Management API."""
        self._ensure_authenticated()
        url = path if path.startswith("https://") else f"{self.BASE_URL}{path}"
        headers = {
            "Authorization": f"Bearer {self._access_token}",
            "Accept": "application/json",
        }
        response = requests.get(url, headers=headers, params=params, timeout=30)
        if not response.ok:
            raise RuntimeError(f"API error {response.status_code}: {response.text[:500]}")
        return response.json()

    def list_flows(self, include_definition: bool = False) -> list[dict]:
        """
        List all flows in the configured environment.

        Endpoint:
            GET /providers/Microsoft.ProcessSimple/environments/{envId}/flows

        Args:
            include_definition: If True, requests the full flow definition JSON.
                                Slower but provides trigger/action details.

        Returns:
            List of flow property dicts.

        Expected response fields per flow:
            name: flow GUID
            properties.displayName: human-readable name
            properties.state: "started" | "stopped" | "suspended"
            properties.createdTime: ISO 8601 timestamp
            properties.lastModifiedTime: ISO 8601 timestamp
            properties.definitionSummary: trigger kind + action count summary
            properties.definition: full flow JSON (if include_definition=True)
        """
        path = (
            f"/providers/Microsoft.ProcessSimple/environments/{self.environment_id}/flows"
        )
        params = {
            "api-version": "2016-11-01",
            "$filter": "search('team')",  # No filter = all flows; adjust as needed
        }
        if include_definition:
            params["$expand"] = "definition"

        all_flows = []
        next_link = None

        while True:
            data = self._get(next_link or path, params=params if not next_link else None)
            all_flows.extend(data.get("value", []))
            next_link = data.get("nextLink")
            if not next_link:
                break

        return all_flows

    def get_flow_runs(
        self,
        flow_name: str,
        top: int = 50,
        status_filter: Optional[str] = None,
    ) -> list[dict]:
        """
        Retrieve run history for a specific flow.

        Endpoint:
            GET /providers/Microsoft.ProcessSimple/environments/{envId}/flows/{flowId}/runs

        Args:
            flow_name: The flow GUID (the `name` field from list_flows, not displayName).
            top: Maximum number of runs to return (default 50, max 250).
            status_filter: Optional filter — 'Succeeded', 'Failed', or 'Running'.

        Returns:
            List of run dicts, each containing:
                name: run GUID
                properties.status: Succeeded | Failed | Running | Cancelled
                properties.startTime: ISO 8601
                properties.endTime: ISO 8601
                properties.code: HTTP status code string for the run
                properties.error: Error details if status is Failed

        Expected output for a succeeded run:
            {
              "name": "08585...",
              "properties": {
                "status": "Succeeded",
                "startTime": "2025-01-08T14:45:01.2Z",
                "endTime": "2025-01-08T14:45:04.8Z"
              }
            }
        """
        path = (
            f"/providers/Microsoft.ProcessSimple/environments/{self.environment_id}"
            f"/flows/{flow_name}/runs"
        )
        params = {"api-version": "2016-11-01", "$top": top}
        if status_filter:
            params["$filter"] = f"properties/status eq '{status_filter}'"

        data = self._get(path, params=params)
        return data.get("value", [])


print("PowerAutomateManagementClient class defined.")

In [ ]:
# Instantiate the Management API client
pa_client = PowerAutomateManagementClient(
    tenant_id=TENANT_ID,
    client_id=CLIENT_ID,
    client_secret=CLIENT_SECRET,
    environment_id=ENVIRONMENT_ID,
)

# Authenticate against the PA Management API scope
# Note: this is a different token than the Graph API token acquired earlier
try:
    pa_client.authenticate()
except RuntimeError as e:
    print(f"[ERROR] {e}")
    print("Ensure the app has been granted access to the Power Automate service.")

In [ ]:
# List flows using the Management API
# Expected: list of dicts with 'name' (GUID) and 'properties' (metadata)

try:
    pa_flows = pa_client.list_flows(include_definition=False)
    print(f"Found {len(pa_flows)} flow(s) via Management API.")
    print()

    # Show a summary table of flows found
    print(f"{'Display Name':<50} {'State':<12} {'Last Modified'}")
    print("-" * 85)
    for flow in pa_flows[:20]:  # Cap display at 20 rows
        props = flow.get("properties", {})
        name = props.get("displayName", "(no name)")[:48]
        state = props.get("state", "unknown")
        modified = props.get("lastModifiedTime", "")[:19]  # Trim subseconds
        print(f"{name:<50} {state:<12} {modified}")

    if len(pa_flows) > 20:
        print(f"... and {len(pa_flows) - 20} more flows.")

except RuntimeError as e:
    print(f"[ERROR] {e}")
    pa_flows = []
    print()
    print("Expected output when working correctly:")
    print(f"{'Display Name':<50} {'State':<12} {'Last Modified'}")
    print("-" * 85)
    print(f"{'Send Teams alert on SharePoint item':<50} {'started':<12} 2025-01-08T14:45:00")
    print(f"{'Weekly exception report':<50} {'started':<12} 2024-12-01T09:00:00")

---

## Section 5: Parsing Flow Metadata

Each flow object from the Management API contains a `properties` dict with rich metadata.
The `definitionSummary` field provides a quick-parse view of the trigger and action types
without needing the full `definition` JSON.

We will write a function that extracts the most useful fields for an audit report.

In [ ]:
def extract_flow_summary(flow: dict) -> dict:
    """
    Extract key metadata from a raw Management API flow object.

    Args:
        flow: Raw flow dict from list_flows(), containing 'name' and 'properties'.

    Returns:
        Flattened summary dict with these keys:
            flow_id: GUID
            display_name: Human-readable name
            state: started | stopped | suspended
            created: ISO 8601 string
            modified: ISO 8601 string
            trigger_kind: The trigger category (e.g., 'ApiConnection', 'Recurrence', 'Request')
            trigger_type: Specific trigger type name if available
            action_count: Total number of actions in the flow
            connector_names: List of unique connector names used
            owner_id: Owner user/service principal GUID
    """
    props = flow.get("properties", {})
    definition_summary = props.get("definitionSummary", {})

    # Extract trigger information from definitionSummary
    triggers = definition_summary.get("triggers", [])
    trigger_kind = triggers[0].get("kind", "unknown") if triggers else "unknown"
    trigger_type = triggers[0].get("type", "unknown") if triggers else "unknown"

    # Extract unique connector names from actions
    actions = definition_summary.get("actions", [])
    connector_names = list({
        action.get("swaggerOperationId", "").split(".")[0]
        for action in actions
        if action.get("swaggerOperationId")
    })

    # Owner information may be under creator or createdBy
    creator = props.get("creator", props.get("createdBy", {}))
    owner_id = creator.get("userId", creator.get("id", "unknown"))

    return {
        "flow_id": flow.get("name", ""),
        "display_name": props.get("displayName", "(no name)"),
        "state": props.get("state", "unknown"),
        "created": props.get("createdTime", "")[:19],
        "modified": props.get("lastModifiedTime", "")[:19],
        "trigger_kind": trigger_kind,
        "trigger_type": trigger_type,
        "action_count": len(actions),
        "connector_names": connector_names,
        "owner_id": owner_id,
    }


# Apply to all flows retrieved
flow_summaries = [extract_flow_summary(f) for f in pa_flows]

if flow_summaries:
    print(f"Parsed {len(flow_summaries)} flow summaries.")
    print()
    # Show detailed view of first flow
    print("=== First Flow Summary ===")
    for key, value in flow_summaries[0].items():
        print(f"  {key:<20} {value}")
else:
    print("No flows to parse. Running with example data for demonstration:")
    example_summary = {
        "flow_id": "abc12345-6789-abcd-ef01-234567890abc",
        "display_name": "Send Teams alert on SharePoint item",
        "state": "started",
        "created": "2024-11-15T10:23:00",
        "modified": "2025-01-08T14:45:00",
        "trigger_kind": "ApiConnection",
        "trigger_type": "OpenApiConnection",
        "action_count": 5,
        "connector_names": ["shared_sharepointonline", "shared_teams"],
        "owner_id": "user-guid-here",
    }
    for key, value in example_summary.items():
        print(f"  {key:<20} {value}")

---

## Section 6: Retrieving and Analyzing Run History

Flow run history is the operational heartbeat of Power Automate. Each run records:
- Start and end time
- Status (Succeeded / Failed / Running / Cancelled)
- Error details for failed runs

Programmatic access to run history enables:
- SLA monitoring (are flows completing within expected time?)
- Failure rate dashboards
- Alert integrations (trigger PagerDuty or Teams notification when failure rate spikes)
- Compliance reporting (evidence that automated processes ran as scheduled)

In [ ]:
def analyze_run_history(runs: list[dict]) -> dict:
    """
    Compute summary statistics from a list of flow run objects.

    Args:
        runs: List of run dicts from get_flow_runs().

    Returns:
        Dict with:
            total_runs: int
            succeeded: int
            failed: int
            running: int
            cancelled: int
            success_rate_pct: float (0-100)
            avg_duration_seconds: float (for completed runs)
            recent_failures: list of (start_time, error_code, error_message) for failed runs
    """
    status_counts: dict[str, int] = {"Succeeded": 0, "Failed": 0, "Running": 0, "Cancelled": 0}
    durations_seconds = []
    recent_failures = []

    for run in runs:
        props = run.get("properties", {})
        status = props.get("status", "Unknown")
        status_counts[status] = status_counts.get(status, 0) + 1

        # Compute duration for completed runs (not Running)
        start_str = props.get("startTime", "")
        end_str = props.get("endTime", "")
        if start_str and end_str and status != "Running":
            try:
                # Parse ISO 8601 timestamps — truncate fractional seconds for compatibility
                start_dt = datetime.fromisoformat(start_str[:19].replace("Z", "+00:00"))
                end_dt = datetime.fromisoformat(end_str[:19].replace("Z", "+00:00"))
                durations_seconds.append((end_dt - start_dt).total_seconds())
            except ValueError:
                pass  # Skip runs with unparseable timestamps

        # Collect failure details for diagnosis
        if status == "Failed":
            error = props.get("error", {})
            recent_failures.append({
                "start_time": start_str[:19],
                "error_code": error.get("code", "unknown"),
                "error_message": error.get("message", "")[:200],
            })

    total = len(runs)
    succeeded = status_counts.get("Succeeded", 0)
    success_rate = (succeeded / total * 100) if total > 0 else 0.0
    avg_duration = sum(durations_seconds) / len(durations_seconds) if durations_seconds else 0.0

    return {
        "total_runs": total,
        "succeeded": succeeded,
        "failed": status_counts.get("Failed", 0),
        "running": status_counts.get("Running", 0),
        "cancelled": status_counts.get("Cancelled", 0),
        "success_rate_pct": round(success_rate, 1),
        "avg_duration_seconds": round(avg_duration, 2),
        "recent_failures": recent_failures[:5],  # Show at most 5 failures
    }


print("analyze_run_history() defined.")

In [ ]:
# Fetch run history for the first flow found (if any)
# Expected: list of run dicts; empty list if flow has no runs yet

if pa_flows:
    first_flow_id = pa_flows[0]["name"]  # 'name' is the GUID, not displayName
    first_flow_name = pa_flows[0]["properties"].get("displayName", "(no name)")
    print(f"Fetching run history for: {first_flow_name}")

    try:
        runs = pa_client.get_flow_runs(flow_name=first_flow_id, top=50)
        print(f"Retrieved {len(runs)} run(s).")
        print()

        stats = analyze_run_history(runs)

        print(f"=== Run History Analysis: {first_flow_name} ===")
        print(f"  Total runs:           {stats['total_runs']}")
        print(f"  Succeeded:            {stats['succeeded']}")
        print(f"  Failed:               {stats['failed']}")
        print(f"  Running:              {stats['running']}")
        print(f"  Success rate:         {stats['success_rate_pct']}%")
        print(f"  Avg duration:         {stats['avg_duration_seconds']}s")

        if stats["recent_failures"]:
            print()
            print("  Recent failures:")
            for failure in stats["recent_failures"]:
                print(f"    {failure['start_time']} | {failure['error_code']} | {failure['error_message'][:80]}")

    except RuntimeError as e:
        print(f"[ERROR] {e}")

else:
    print("No flows available to fetch runs for.")
    print()
    print("Expected output when working correctly:")
    print("=== Run History Analysis: Send Teams alert on SharePoint item ===")
    print("  Total runs:           47")
    print("  Succeeded:            44")
    print("  Failed:               3")
    print("  Running:              0")
    print("  Success rate:         93.6%")
    print("  Avg duration:         4.21s")

---

## Exercise: Build a Flow Audit Report

**Task:** Write a function `generate_audit_report(pa_client)` that:

1. Lists all flows in the environment
2. For each flow, retrieves the last 20 runs
3. Computes success rate and average duration using `analyze_run_history()`
4. Returns a list of dicts sorted by success rate (lowest first) — highlighting the flows that need attention
5. Prints a formatted report table

**Hints:**
- Use `pa_client.list_flows()` and `pa_client.get_flow_runs(flow_name, top=20)`
- Use `extract_flow_summary()` to parse flow metadata
- Sort the result list with `sorted(..., key=lambda x: x['success_rate_pct'])`
- Handle the case where a flow has zero runs (success rate = None, not 0%)

**Expected output (example):**

```
=== Flow Audit Report — 2025-01-08 ===
Environment: my-org-production

Flow Name                              State    Runs  Succeed%  Avg Duration
-----------------------------------------------------------------------
Invoice OCR processor                  started    20     65.0%      12.4s
Weekly SQL exception report            started    20     85.0%       3.1s
SharePoint → Teams notification        started    20    100.0%       1.8s
```

In [ ]:
# YOUR IMPLEMENTATION HERE
# Replace the pass statement with your code

def generate_audit_report(client: PowerAutomateManagementClient, runs_per_flow: int = 20) -> list[dict]:
    """
    Generate a flow health audit report for the environment.

    Args:
        client: Authenticated PowerAutomateManagementClient.
        runs_per_flow: Number of recent runs to analyze per flow.

    Returns:
        List of audit record dicts sorted by success_rate_pct ascending.
        Each record has: display_name, state, total_runs, success_rate_pct, avg_duration_seconds
    """
    # Step 1: List all flows
    # Step 2: For each flow, get run history
    # Step 3: Analyze runs using analyze_run_history()
    # Step 4: Build audit record dicts
    # Step 5: Sort by success_rate_pct ascending
    pass  # Replace with your implementation


# Run your function when ready:
# audit = generate_audit_report(pa_client)
print("Implement generate_audit_report() above, then uncomment the line below to run it.")
# audit_records = generate_audit_report(pa_client)

In [ ]:
# Reference solution — study this after attempting the exercise yourself

def generate_audit_report_solution(
    client: PowerAutomateManagementClient,
    runs_per_flow: int = 20,
) -> list[dict]:
    """
    Complete audit report implementation.
    Lists all flows, fetches recent runs, computes health metrics,
    and returns results sorted by success rate ascending.
    """
    audit_records = []

    # Step 1: Get all flows in the environment
    all_flows = client.list_flows(include_definition=False)
    print(f"Auditing {len(all_flows)} flow(s)...")

    for flow in all_flows:
        summary = extract_flow_summary(flow)
        flow_id = flow["name"]

        # Step 2: Fetch run history (handle flows with no runs gracefully)
        try:
            runs = client.get_flow_runs(flow_name=flow_id, top=runs_per_flow)
        except RuntimeError:
            # Some flows may not have run history accessible (permissions, etc.)
            runs = []

        # Step 3: Analyze runs
        stats = analyze_run_history(runs)

        # Step 4: Build audit record — None for success_rate when there are no runs
        audit_records.append({
            "display_name": summary["display_name"],
            "state": summary["state"],
            "total_runs": stats["total_runs"],
            "success_rate_pct": stats["success_rate_pct"] if stats["total_runs"] > 0 else None,
            "avg_duration_seconds": stats["avg_duration_seconds"],
            "recent_failures": stats["recent_failures"],
        })

    # Step 5: Sort by success_rate_pct ascending; flows with no runs go last
    sorted_records = sorted(
        audit_records,
        key=lambda r: (r["success_rate_pct"] is None, r["success_rate_pct"] or 0),
    )

    # Print formatted report
    today = datetime.now(timezone.utc).strftime("%Y-%m-%d")
    print()
    print(f"=== Flow Audit Report — {today} ===")
    print()
    print(f"{'Flow Name':<45} {'State':<10} {'Runs':<6} {'Succeed%':<10} {'Avg Duration'}")
    print("-" * 85)
    for rec in sorted_records:
        rate = f"{rec['success_rate_pct']}%" if rec["success_rate_pct"] is not None else "N/A"
        duration = f"{rec['avg_duration_seconds']}s" if rec["avg_duration_seconds"] else "N/A"
        print(
            f"{rec['display_name'][:43]:<45} "
            f"{rec['state']:<10} "
            f"{rec['total_runs']:<6} "
            f"{rate:<10} "
            f"{duration}"
        )

    return sorted_records


# Uncomment to run the solution audit:
# audit_records = generate_audit_report_solution(pa_client)
print("Reference solution defined. Uncomment the last line to run it against your environment.")

---

## Summary

### Key Takeaways

1. **Two API surfaces exist** for Power Automate: the Microsoft Graph API (`graph.microsoft.com`) and the Power Automate Management API (`api.flow.microsoft.com`). The Management API is more complete for flow operations; the Graph API is broader for Microsoft 365 integration.

2. **OAuth 2.0 client credentials** (not user login) is the correct auth pattern for programmatic/daemon access. Each API surface requires its own scope: `https://graph.microsoft.com/.default` for Graph, `https://service.flow.microsoft.com/.default` for Management API.

3. **Environment ID** scopes all flow operations. Get it from the portal URL or the Session details page. Every tenant operation should specify an environment — the Default environment is not automatically assumed by the API.

4. **Run history** is the operational data layer for Power Automate. Programmatic analysis of run history enables monitoring, alerting, and compliance reporting at scale — things the portal UI does not support well for large tenants.

5. **Pagination** is required for large tenants. Both APIs return `@odata.nextLink` / `nextLink` tokens for multi-page collections. Always use a paginating loop rather than assuming all results fit in the first response.

### What's Next

Module 01 moves to hands-on flow building: you will create a complete automated cloud flow end-to-end using the portal — with a SharePoint trigger, data operations, conditional branching, and a Teams notification action. The API knowledge from this notebook gives you context for what the portal is configuring under the hood.

### Additional Resources

- [Microsoft Graph API — flows reference](https://learn.microsoft.com/en-us/graph/api/resources/flow) — current endpoint documentation
- [Power Automate Management API](https://learn.microsoft.com/en-us/power-automate/web-api) — full Management API reference
- [Azure AD app registration](https://learn.microsoft.com/en-us/azure/active-directory/develop/quickstart-register-app) — step-by-step app registration guide
- [Power Platform CLI](https://learn.microsoft.com/en-us/power-platform/developer/cli/introduction) — `pac flow list` and related commands offer a CLI alternative to direct API calls

In [ ]:
key_takeaways([
    "Review the key concepts covered in this notebook",
    "Practice applying these techniques to your own data",
    "Connect this material to the companion guide and slides"
])